# Session 10 · Applied Prompting, Fine-Tuning Basics and Soft Prompts

Generative AI › Track 1 · Prompt Engineering

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-10_finetune_dataset.py`

In [1]:
"""Builds and validates a small fine-tuning dataset in chat-JSONL format.
Each line = one example: system + user + assistant messages. Bad rows are reported, not silently kept."""
import json

SYSTEM = "You are IoBM Helpdesk. Answer in at most 2 sentences, politely, in simple English."
raw = [
    ("How do I reset my LMS password?", "Click 'Forgot password' on the LMS login page and follow the email link. If no email arrives in 10 minutes, contact IT at the helpdesk counter."),
    ("When is the add/drop deadline?", "The add/drop deadline is published in the academic calendar on the student portal. Please check it there, as dates change each semester."),
    ("Can I bring my own laptop to the lab?", "Yes, you can use your own laptop in the lab. Connect to the campus Wi-Fi with your student ID."),
    ("", "This row has no question."),
    ("Where is the library?", "The library is on the ground floor of the main block, next to the cafeteria, and it is open from 8 am to 8 pm on weekdays. Bring your student card to borrow books. Laptops are allowed in the reading area. Group rooms can be booked online."),
]

def valid(q, a):
    if not q.strip(): return "empty question"
    if a.count(".") > 2: return "answer longer than 2 sentences (breaks the style rule)"
    return None

good, report = [], []
for i, (q, a) in enumerate(raw, 1):
    problem = valid(q, a)
    if problem:
        report.append(f"row {i}: REJECTED ({problem})")
        continue
    good.append({"messages": [{"role": "system", "content": SYSTEM},
                              {"role": "user", "content": q},
                              {"role": "assistant", "content": a}]})
    report.append(f"row {i}: ok")

with open("session-10_helpdesk_train.jsonl", "w") as f:
    for ex in good:
        f.write(json.dumps(ex) + "\n")

print("\n".join(report))
print(f"\nKept {len(good)} of {len(raw)} examples -> session-10_helpdesk_train.jsonl")
print("First line:\n" + json.dumps(good[0])[:160] + "...")

row 1: ok
row 2: ok
row 3: ok
row 4: REJECTED (empty question)
row 5: REJECTED (answer longer than 2 sentences (breaks the style rule))

Kept 3 of 5 examples -> session-10_helpdesk_train.jsonl
First line:
{"messages": [{"role": "system", "content": "You are IoBM Helpdesk. Answer in at most 2 sentences, politely, in simple English."}, {"role": "user", "content": "...


### Example · `session-10_soft_prompt_toy.py`

In [2]:
"""Toy 'soft prompt' (prompt tuning) demo in pure Python.
A frozen 'model' adds up each word's sentiment (x), weighting each word by how much it 'attends'
to the prompt vector: weight = 1 + soft . v. The word vectors and the scoring rule never change.
We learn ONLY the 2-number soft prompt so the frozen model stops being fooled by
complaints that start with a polite word ("great, we waited hours again")."""
VEC = {"great": (1.0, -0.5), "love": (0.9, -0.5), "fast": (0.6, 0.0), "service": (0.1, 0.0),
       "waited": (-0.2, 0.9), "hours": (-0.1, 0.8), "broken": (-0.9, 0.3), "again": (0.0, 0.4)}
data = [("great fast service", 1), ("love the service", 1),
        ("great waited hours again", 0), ("love waited hours", 0), ("broken again", 0)]

def model(sentence, soft):
    score = 0.0
    for w in sentence.split():
        x, y = VEC.get(w, (0.0, 0.0))
        score += x * (1 + soft[0] * x + soft[1] * y)
    return score

def accuracy(soft):
    return sum((model(s, soft) > 0) == bool(l) for s, l in data) / len(data)

def loss(soft):  # hinge loss with margin 0.1
    return sum(max(0.0, 0.1 - (1 if l else -1) * model(s, soft)) for s, l in data)

soft = [0.0, 0.0]
print(f"Before tuning: soft prompt = ({soft[0]:+.2f}, {soft[1]:+.2f}), accuracy = {accuracy(soft):.0%}")
lr, eps = 0.1, 1e-4
for step in range(300):  # numeric gradient descent; only the soft prompt moves, the model stays frozen
    grad = []
    for k in range(2):
        up = soft.copy(); up[k] += eps
        grad.append((loss(up) - loss(soft)) / eps)
    soft = [soft[k] - lr * grad[k] for k in range(2)]
print(f"After tuning : soft prompt = ({soft[0]:+.2f}, {soft[1]:+.2f}), accuracy = {accuracy(soft):.0%}, loss = {loss(soft):.2f}")
for s, l in data:
    print(f"  {s:26} label={'pos' if l else 'neg'}  score={model(s, soft):+.2f}")

Before tuning: soft prompt = (+0.00, +0.00), accuracy = 60%
After tuning : soft prompt = (-0.57, +0.44), accuracy = 100%, loss = 0.00
  great fast service         label=pos  score=+0.69
  love the service           label=pos  score=+0.33
  great waited hours again   label=neg  score=-0.24
  love waited hours          label=neg  score=-0.21
  broken again               label=neg  score=-1.48
